In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "code").is_dir() and (path / "model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing code/ and model/")
DATA_DIR = PROJECT_ROOT / "code" / "data"
MODEL_DIR = PROJECT_ROOT / "model"
ASSET_DIR = PROJECT_ROOT / "src"



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load artifacts
best_xgb  = joblib.load(MODEL_DIR / "xgb_best_model.pkl")
scaler    = joblib.load(MODEL_DIR / "scaler.pkl")
feat_names = joblib.load(MODEL_DIR / "feature_names.pkl")

# Load processed data
df = pd.read_csv(DATA_DIR / "Edited_Data.csv")
X  = df.drop('Churn', axis=1)
y  = df['Churn']

X_scaled_df = pd.DataFrame(scaler.transform(X), columns=feat_names)
churn_probs = best_xgb.predict_proba(X_scaled_df)[:, 1]
df['churn_probability'] = churn_probs
df['predicted_churn']   = (churn_probs >= 0.5).astype(int)

print(f"Dataset loaded: {df.shape}")
print(f"Overall churn rate: {y.mean()*100:.1f}%")
print(f"Customers predicted to churn: {df['predicted_churn'].sum()} ({df['predicted_churn'].mean()*100:.1f}%)")

Dataset loaded: (7043, 51)
Overall churn rate: 26.5%
Customers predicted to churn: 1434 (20.4%)


Steps:


a. Model predicts high-risk customers
              
b. Identify who needs intervention
              
c. Choose a retention action
              
d. Estimate potential revenue saved

In [3]:
#churn probability for all customers
churn_prob = best_xgb.predict_proba(X_scaled_df)[:,1]

#creating risk categories based on churn probability
df['risk_group'] = pd.cut(df['churn_probability'],bins = [0, 0.3, 0.6, 1], labels = ('low', 'medium', 'high'))
print(df['risk_group'].value_counts())



risk_group
low       4387
medium    1745
high       911
Name: count, dtype: int64


In [4]:
# finding the high risk customers and their important features
high_risk_customers = df[df['risk_group'] == 'high']

# important features for high risk customers
high_risk_customers[['tenure', 'MonthlyCharges', 'churn_probability', 'Churn']]


,tenure,MonthlyCharges,churn_probability,Churn
0,1,29.85,0.687027,0
4,2,70.70,0.641818,1
5,8,99.65,0.833987,1
20,1,39.65,0.750977,1
31,2,95.50,0.743761,0
...,...,...,...,...
7008,3,75.05,0.656343,1
7010,1,74.45,0.864184,1
7018,1,70.65,0.745047,1
7032,1,75.75,0.878112,1


In [5]:
# Calculating revenue at risk
monthly_revenue_at_risk = high_risk_customers['MonthlyCharges'].sum()

print(f"Monthly revenue at risk: ${monthly_revenue_at_risk}")

# annual revenue at risk
annual_revenue_at_risk = monthly_revenue_at_risk * 12

print(
    f"Annual revenue at risk: "
    f"${annual_revenue_at_risk:,.0f}"
)

Monthly revenue at risk: $73007.6
Annual revenue at risk: $876,091


In [6]:
#Choosing Intervention
#Based on SHAP analysis, month-to-month contract customers are at higher risk, prone to churn. 
# We should consider offering them incentives to stay with the company, such as discounts or loyalty rewards.

target_customers = df[
    (df['churn_probability'] >= 0.60) &
    (df['Contract_Month-to-month'] == 1)
]

print("Customers targeted:", len(target_customers))

Customers targeted: 911


In [7]:
print(
    target_customers[
        ['tenure',
         'MonthlyCharges',
         'churn_probability',
         'Churn']
    ].head(10)
)

    tenure  MonthlyCharges  churn_probability  Churn
0        1           29.85           0.687027      0
4        2           70.70           0.641818      1
5        8           99.65           0.833987      1
20       1           39.65           0.750977      1
31       2           95.50           0.743761      0
36       5           69.70           0.631623      1
39      11           97.85           0.743001      1
47       2           80.65           0.718433      1
53       8           80.65           0.637985      1
55      18           95.45           0.659908      1


In [13]:
# Estimate potential revenue loss from targeted customers
# If the intervention retains 20% of the targeted customers, what would the potential financial impact be?

retention_rate = 0.20

#customer retentained
customers_saved = int(
    len(target_customers) * retention_rate
)
print("Estimated number of customers saved:", customers_saved)

# Estimate potential monthly revenue saved
revenue_saved = (
    target_customers['MonthlyCharges'].sum()
    * retention_rate
)
print("Estimated monthly revenue saved: ", f"${revenue_saved:,.0f}")

# Estimate potential annual revenue saved
potential_annual_revenue_saved = (
    revenue_saved * 12
)

print(
    f"Potential annual revenue saved: "
    f"${potential_annual_revenue_saved:,.0f}")

Estimated number of customers saved: 182
Estimated monthly revenue saved:  $14,602
Potential annual revenue saved: $175,218
